<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex10.2-solid-oxide-cell/Ex10.2_00_cell_lab.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_10.2 · Notebook 00 — the device

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L10.2 (Solid oxide cells)**

## What Ex_10.2 is about

A reversible solid oxide cell in **both modes**: as a fuel cell (SOFC) it
turns hydrogen into electricity, and as an electrolyser (SOEC) it turns
electricity and steam into hydrogen. The hardware is the same either way; the
sign of the current selects the mode and nothing else changes. You fit a 0-D
button-cell model, solve a 1-D gas channel with a physics-informed network,
compute how temperature trades production against lifetime, and finish by
optimising a 24-hour operating trajectory through the cell model itself.

There is no community reference implementation to lean on. Every parameter
carries a source tag, and values marked `ESTIMATED` are order-of-magnitude
placeholders, not measurements. Knowing which of your results are defensible
is what this set is built around.

### Goals

By the end of the exercise set you can

1. read a polarisation curve continuously through zero current, and say why
   the three overpotentials subtract in fuel-cell mode and add in electrolysis;
2. locate the thermoneutral point from $q = i(V - V_{tn})$ and predict the sign
   of the heat either side of it;
3. fit a 0-D button-cell model to a noisy polarisation curve and verify it
   against four independent checks rather than against its own residual;
4. write a 1-D convection–diffusion channel residual, and quantify the error a
   0-D model makes as reactant utilisation rises;
5. produce the temperature trade-off — production rate against lifetime — as a
   computed result, and test how far it moves with an `ESTIMATED` parameter;
6. optimise an operating trajectory by differentiating through the cell model,
   check it against brute force, and state which of your numbers you would
   defend.

### Method — six notebooks, run in order

| notebook | what you do | problem / data | lecture |
|---|---|---|---|
| **00** | watch the device in both directions; nothing to write | polarisation curve, thermoneutral point, temperature, channel | L10.2 |
| **01** | fit a 0-D model and pass four verification checks | a synthetic, noisy button-cell polarisation curve generated by the model itself | L10.2 |
| **02** | a PINN for the gas channel, and the error of 0-D as utilisation rises | 1-D convection–diffusion along the channel | L10.2 |
| **03** | the temperature trade-off as a computed curve, and the temperature that maximises hydrogen over life | production rate and lifetime to end of life against temperature, at 1 A/cm² | L10.2 |
| **04** | lifetime-aware optimisation of a 24-hour trajectory | the cell model, differentiated through; brute force as a check | L10.2 |
| **05** | the report | the results saved in `Ex10.2_outputs/` | — |

Later notebooks load results saved by earlier ones, so run them in order.

### Applications

* **Power-to-hydrogen and back.** One stack that produces hydrogen when power
  is cheap and generates electricity when it is needed. Sign convention
  throughout: **i > 0 is electrolysis (SOEC), i < 0 is fuel cell (SOFC)**.
* **Thermal operation.** Near the thermoneutral voltage (about 1.29 V for
  steam at 800 °C) an electrolyser neither needs nor rejects heat, which is
  why it is a standard operating point.
* **Lifetime-aware operation.** Higher temperature lowers the area-specific
  resistance **and** raises the degradation rate. Notebook 04 makes an
  optimiser choose, against a constant-current baseline and an end-of-life
  constraint.

## What this notebook does

**Read and run; you are not asked to rewrite this.**

One cell, two directions. Negative current is fuel-cell mode, positive is
electrolysis, and the polarisation curve passes smoothly through the
open-circuit voltage between them.

### A warning about the reference

**There is no PyBaMM here** — no community implementation, no canonical
parameter file. `problem.py` is a course reference, and every parameter
carries a source tag. Values marked `ESTIMATED` are order-of-magnitude
placeholders chosen to reproduce published behaviour. Any result depending on
them must say so.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex10.2-solid-oxide-cell/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · The polarisation curve, both directions

In [ ]:
print("modes:", pb.MODES)

soec = pb.SOCParams(mode="SOEC", T=1073.15, p_H2=0.10, p_H2O=0.90)
sofc = pb.SOCParams(mode="SOFC", T=1073.15, p_H2=0.90, p_H2O=0.10)
print(soec, "\n", sofc)
print(f"\nOCV in SOEC gas (10/90 H2/H2O): {pb.nernst(soec):.4f} V")
print(f"OCV in SOFC gas (90/10 H2/H2O): {pb.nernst(sofc):.4f} V")
print(f"thermoneutral voltage         : {pb.thermoneutral_voltage():.4f} V")
pb.plot_polarisation([soec, sofc], ["SOEC gas (10/90)", "SOFC gas (90/10)"])

Note that the OCV depends on gas composition, not on which mode you intend to
run — the curve is one continuous function of current.

## 2 · The thermoneutral point

Heat generation is $q = i(V - V_{tn})$: negative below the thermoneutral
voltage (the cell absorbs heat), zero at it, positive above.

In [ ]:
pb.plot_heat(soec)
i, V = pb.polarisation_curve(soec)
k = int(np.argmin(np.abs(V - pb.thermoneutral_voltage())))
print(f"heat vanishes at i = {i[k]:+.3f} A/cm2, V = {V[k]:.4f} V")
print(f"thermoneutral voltage         = {pb.thermoneutral_voltage():.4f} V")

## 3 · Temperature is the main lever

In [ ]:
pars = [pb.SOCParams("SOEC", T + 273.15, 0.10, 0.90) for T in (700, 750, 800, 850)]
pb.plot_polarisation(pars, [f"{p.T_celsius:.0f} C" for p in pars])
for p_ in pars:
    print(f"  {p_.T_celsius:.0f} C:  OCV {pb.nernst(p_):.3f} V   "
          f"degradation rate at 1 A/cm2 = {pb.degradation_rate(1.0, p_):.3e}")

Higher temperature lowers the resistance **and** raises the degradation rate.
That is the trade-off L10.2 builds its optimisation on, visible in two lines of output.

## 4 · Along the channel

In [ ]:
pb.plot_channel(soec, i_mean=1.0)

---

## 5 · Ready

You have seen the device in both directions, the thermoneutral point, the
temperature lever and the composition gradient along the channel. Nothing
below this point in Ex_10.2 depends on anything you have not just watched
happen.

Next: **[notebook 01](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex10.2-solid-oxide-cell/Ex10.2_01_button_cell.ipynb)**, where you fit the 0-D button cell to a polarisation
curve and verify the fit against four checks.